# Half an AMD GPU in a Run:ai workspace
From the guide: https://manmohanbrahma.github.io/ai/runai-nvidia-amd/#demo

## Cell 1: who am I, and what share did Run:ai give me

In [ ]:
import os
for k in ("RUNAI_PROJECT", "RUNAI_JOB_NAME", "OIF_GPU_PCI", "OIF_GPU_MEMORY_FRACTION", "PYTORCH_ALLOC_CONF"):
    print(f"{k:25} {os.environ.get(k)}")

## Cell 2: which GPU this container can reach

In [ ]:
import os, torch
print("device files:", sorted(os.listdir("/dev/dri")), "| /dev/kfd:", os.path.exists("/dev/kfd"))
p = torch.cuda.get_device_properties(0)
print(f"{p.name} | {p.gcnArchName} | {p.multi_processor_count} compute units | {p.total_memory/2**30:.1f} GiB physical")
print("PyTorch", torch.__version__, "| HIP", torch.version.hip)

## Cell 3: the card's memory and this pod's share

In [ ]:
import os, torch
free, total = torch.cuda.mem_get_info()
share = float(os.environ.get("OIF_GPU_MEMORY_FRACTION", "1"))
print(f"GPU memory {total/2**30:.1f} GiB, free {free/2**30:.1f} GiB, this pod's share {share} = {share*total/2**30:.1f} GiB")
for gib in (20, 25):
    try:
        x = torch.empty(int(gib * 2**30) // 2, dtype=torch.float16, device="cuda"); print(f"{gib} GiB: allocated"); del x
    except torch.OutOfMemoryError:
        print(f"{gib} GiB: refused (over the share)")
torch.cuda.empty_cache()

## Cell 4: what the driver sees

In [ ]:
!amd-smi static --gpu 0 --asic --vram | head -30
!amd-smi metric --gpu 0 --usage --mem-usage

## Cell 5: how fast is this half (30 seconds)

In [ ]:
import time, torch
a = torch.randn(8192, 8192, device="cuda", dtype=torch.float16); torch.cuda.synchronize()
t, n = time.time(), 0
while time.time() - t < 30: a @ a; n += 1
torch.cuda.synchronize(); print(f"fp16 matmul: {n * 2 * 8192**3 / (time.time() - t) / 1e12:.1f} TFLOPS")